# 🧠 AI Logic Engine: Main Inference Console

This notebook allows you to perform high-precision symbolic reasoning against your Firestore Knowledge Graph. It mirrors the reasoning logic of the main application but runs in a high-performance Python environment.

---

In [ ]:
# @title 🛠️ Step 1: Initialize Logic Engine
!pip install -q firebase-admin

from google.colab import files
import firebase_admin
from firebase_admin import credentials, firestore

# @markdown 1. Go to [Firebase Console](https://console.firebase.google.com/project/gen-lang-client-0802662324/settings/serviceaccounts/adminsdk)
# @markdown 2. Click **"Generate new private key"** and save the JSON file.
# @markdown 3. Run this cell to upload that file directly to Colab.

print("Please upload your serviceAccountKey.json file:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

DATABASE_ID = 'ai-studio-09d97652-b1c3-4b1a-b63e-31d8a38be4c2' 

if not firebase_admin._apps:
    cred = credentials.Certificate(filename)
    firebase_admin.initialize_app(cred)

db = firestore.client(database_id=DATABASE_ID)
print(f"✅ Logic Engine Memory Initialized for Database: {DATABASE_ID}")

In [ ]:
# @title ⚙️ Step 2: Symbolic Inference Engine (Python Core)

class PythonLogicEngine:
    def __init__(self, db):
        self.db = db
        self.cache = {}

    def get_node(self, node_id):
        key = node_id.lower().strip()
        if key in self.cache: return self.cache[key]
        
        doc = self.db.collection('nodes').document(key).get()
        if doc.exists:
            data = doc.to_dict()
            self.cache[key] = data
            return data
        return None

    def find_path(self, start, end, max_depth=6):
        queue = [(start, [], 1.0)]
        visited = {start.lower()}
        
        while queue:
            node_name, path, certainty = queue.pop(0)
            
            if node_name.lower() == end.lower():
                return path, certainty
            
            if len(path) >= max_depth: continue
            
            node = self.get_node(node_name)
            if not node: continue
            
            # Process Relations
            for rel in node.get('relations', []):
                target = rel['targetId']
                if target.lower() not in visited:
                    new_path = path + [f"{node_name} {rel['verb']} {target}"]
                    queue.append((target, new_path, certainty * 0.9))
            
            # Process Inheritance
            for group in node.get('groups', []):
                if group.lower() not in visited:
                    new_path = path + [f"{node_name} is_a {group}"]
                    queue.append((group, new_path, certainty * 0.99))
        
        return None, 0

engine = PythonLogicEngine(db)
print("✅ Inference Core Loaded.")

In [ ]:
# @title 🔍 Step 3: Test Reasoning (Query Console)
SUBJECT = "Socrates" # @param {type:"string"}
PREDICATE = "mortal" # @param {type:"string"}

path, score = engine.find_path(SUBJECT, PREDICATE)

if path:
    print(f"✅ Logical Proof Found (Certainty: {score*100:.1f}%)")
    for i, step in enumerate(path):
        print(f"  Step {i+1}: {step}")
else:
    print("❌ No logical connection found in the database.")